# What this notebook is doing (temperature summary stats)

This notebook computes summary air-temperature metrics across the datasets I have been using for Skagit.
It is basically the temperature version of the precipitation summary notebook, but focused on the three Skagit sub-basins.

It pulls daily temperature information for each dataset and then calculates region-based annual and summary metrics like annual average temperature, average daily min and max, summer max conditions, and counts of hot or freezing days.
The regions used here are Upper Skagit, Sauk, and Lower Skagit.

At the end it writes annual metrics, summary metrics, and region-specific summary tables so the temperature comparisons can be reused outside the notebook.


In [ ]:
# ============================================================
# NOTEBOOK: Skagit air-temperature summary metrics across datasets
#
# Computes, for each dataset x region:
#   1) Annual Average (°F)
#   2) Annual Average Daily Min (°F)
#   3) Annual Average Daily Max (°F)
#   4) Average Summer (June - Aug) Max Temp (°F)
#   5) Annual Average Days with Max Temp Above 86°F
#   6) Annual Average Days with Min Temp Below 32°F
#
# YEAR CONVENTION:
#   Water year label Y = Oct(Y-1) .. Sep(Y)
#
# IMPORTANT:
# - Uses the same 3 regions as your current temperature notebook
# - Datasets with only daily mean temperature will have NaN for:
#       Annual Average Daily Min
#       Annual Average Daily Max
#       Summer Max
#       Days > 86°F
#       Days < 32°F
# - HRRR06 and PNNL use native subdaily temperature and are resampled
#   to daily mean/min/max AFTER basin-mean aggregation
# ============================================================


import re
import glob
import math
import time
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- Config ----------------
BASE = Path("/data0/balaji24/data")
BOUNDARY_GEO = Path("../data/GIS/SkagitBoundary.json")
HUC8_GEO     = Path("../data/GIS/SkagitSubBasin_HUC8.geojson")

YEAR_MIN, YEAR_MAX = 1980, 2024
GLOBAL_START = f"{YEAR_MIN-1}-10-01"
GLOBAL_END   = f"{YEAR_MAX}-09-30"

REGION_NAMES = ["Upper Skagit", "Sauk", "Lower Skagit"]

# Local datasets
P_HRRR_DIR  = BASE / "weather_data_06"
PNNL_HIST   = BASE / "PNNL" / "historical"
PNNL_GEO    = PNNL_HIST / "SERDP6km.geo_em.d01.nc"

UCLA_T2_GLOB    = "/data0/balaji24/data/ucla_era5_d02_daily/t2/t2.daily.era5.d02.*.nc"
UCLA_COORD_FILE = Path("/data0/balaji24/data/ucla_era5_d02_daily/static/wrfinput_d02_coord.nc")

DAYMET_DIR       = "/data0/balaji24/data/daymet_nc_2014_2024"
DAYMET_TMIN_GLOB = f"{DAYMET_DIR}/daymet_tmin_*.nc"
DAYMET_TMAX_GLOB = f"{DAYMET_DIR}/daymet_tmax_*.nc"

PRISM_TEMP_DIR = Path("/data0/balaji24/data/prism_temp_nc")
PRISM_ZIP_GLOB = "prism_tmean_us_25m_*.zip"

# Dataset-specific exclusions from your current notebook
DAYMET_SKIP_WY = {2023}
HRRR_SKIP_WY   = {2014}

# Coverage thresholds
MIN_DAYS_PER_WY = 330
MIN_SUMMER_DAYS = 75

RUN_DATASETS = {
    "DAYMET":   True,
    "PRISM":    True,
    "UCLA":     True,
    "HRRR06":   True,
    "PNNL":     True,
    "CONUS404": True,
}

OUT = BASE / "derived" / "temperature_summary_metrics"
OUT.mkdir(parents=True, exist_ok=True)

ANNUAL_CSV  = OUT / f"temp_annual_metrics_{YEAR_MIN}_{YEAR_MAX}.csv"
SUMMARY_CSV = OUT / f"temp_summary_metrics_{YEAR_MIN}_{YEAR_MAX}.csv"


def _ensure_time_name(obj):
    if isinstance(obj, xr.Dataset):
        if "time" not in obj.dims:
            for alt in ("day", "date"):
                if alt in obj.dims:
                    obj = obj.rename({alt: "time"})
                    break
        if "time" not in obj.coords:
            for alt in ("day", "date"):
                if alt in obj.coords:
                    obj = obj.rename({alt: "time"})
                    break
        return obj

    if isinstance(obj, xr.DataArray):
        if "time" not in obj.dims:
            for alt in ("day", "date"):
                if alt in obj.dims:
                    obj = obj.rename({alt: "time"})
                    break
        return obj

    return obj

def ensure_time_sorted_unique(da):
    da = _ensure_time_name(da)
    if "time" not in da.dims:
        return da
    da = da.sortby("time")
    try:
        t = pd.to_datetime(da["time"].values)
        _, idx = np.unique(t, return_index=True)
        da = da.isel(time=np.sort(idx))
    except Exception:
        pass
    return da

def to_celsius(da):
    da = _ensure_time_name(da)

    da = xr.where(np.isfinite(da), da, np.nan)
    da = xr.where(np.abs(da) < 1e20, da, np.nan)

    try:
        probe = da.isel(time=0) if "time" in da.dims else da
        m = float(probe.mean(skipna=True).compute()) if hasattr(probe.data, "compute") else float(probe.mean(skipna=True))
    except Exception:
        m = 0.0

    out = da - 273.15 if m > 150 else da
    out = xr.where(np.isfinite(out), out, np.nan)
    out = xr.where((out > -80) & (out < 80), out, np.nan)
    return out

def c_to_f(x):
    return (x * 9.0 / 5.0) + 32.0

def find_lat_lon_names(ds):
    lat_name = next((n for n in ["lat","latitude","lat2d","XLAT","XLAT_M","Lat","LAT"] if n in ds), None)
    lon_name = next((n for n in ["lon","longitude","lon2d","XLONG","XLONG_M","Lon","LON"] if n in ds), None)
    return lat_name, lon_name

def _to_2d(arr):
    arr = _ensure_time_name(arr)
    if "time" in arr.dims:
        arr = arr.isel(time=0)
    for d in list(arr.dims):
        if arr.sizes[d] == 1:
            arr = arr.isel({d: 0})
    while arr.ndim > 2:
        extra = [d for d in arr.dims if d not in ("x","y","lon","lat","lon2d","lat2d","south_north","west_east")]
        if not extra:
            break
        arr = arr.isel({extra[0]: 0})
    return arr

def water_year_index(idx):
    idx = pd.DatetimeIndex(idx)
    wy = idx.year.to_numpy().copy()
    wy[idx.month >= 10] += 1
    return pd.Index(wy, name="year")

def water_year_index_xr(time_da):
    wy = xr.where(time_da.dt.month >= 10, time_da.dt.year + 1, time_da.dt.year)
    return wy.rename("year")

def filter_year_range(series: pd.Series) -> pd.Series:
    if series.empty:
        return series
    return series[(series.index >= YEAR_MIN) & (series.index <= YEAR_MAX)]

def load_regions_gdf():
    import geopandas as gpd

    skagit_gdf = gpd.read_file(BOUNDARY_GEO).to_crs("EPSG:4326")
    huc8 = gpd.read_file(HUC8_GEO).to_crs("EPSG:4326")

    huc8_3 = huc8[huc8["Name"].isin(REGION_NAMES)].copy()
    huc8_3 = gpd.overlay(huc8_3, skagit_gdf, how="intersection")

    regions_gdf = huc8_3[["Name", "geometry"]].reset_index(drop=True)
    regions_gdf["Name"] = pd.Categorical(regions_gdf["Name"], categories=REGION_NAMES, ordered=True)
    return regions_gdf.sort_values("Name").reset_index(drop=True)

def region_mean(da, grid_ds, regions_gdf):
    """
    Robust region mean using regionmask.
    Handles lon/lat as 2D or 1D (meshgrid) and aligns dims to da.
    """
    import regionmask

    da = _ensure_time_name(da)
    grid_ds = _ensure_time_name(grid_ds)

    time_dim = "time" if "time" in da.dims else None
    spatial = [d for d in da.dims if d != time_dim]
    if not spatial:
        return da.expand_dims(region=list(regions_gdf["Name"].values))

    lat_name, lon_name = find_lat_lon_names(grid_ds)
    if not (lat_name and lon_name):
        out = da.mean(spatial, skipna=True)
        return out.expand_dims(region=list(regions_gdf["Name"].values))

    lon = _to_2d(grid_ds[lon_name])
    lat = _to_2d(grid_ds[lat_name])

    if lon.ndim == 1 and lat.ndim == 1 and len(spatial) == 2:
        lon_dim = next((d for d in spatial if da.sizes.get(d, -1) == lon.size), None)
        lat_dim = next((d for d in spatial if da.sizes.get(d, -1) == lat.size and d != lon_dim), None)
        if lon_dim and lat_dim:
            lon2d, lat2d = np.meshgrid(lon.values, lat.values)
            lon = xr.DataArray(lon2d, dims=(lat_dim, lon_dim))
            lat = xr.DataArray(lat2d, dims=(lat_dim, lon_dim))

    regs = regionmask.Regions(
        outlines=list(regions_gdf.geometry.values),
        names=list(regions_gdf["Name"].astype(str).values),
        numbers=list(range(len(regions_gdf))),
        name="Skagit_3HUC8",
    )
    rid = regs.mask(lon, lat)

    if set(rid.dims) == set(spatial) and list(rid.dims) != spatial:
        rid = rid.transpose(*spatial)

    out_list = []
    for i, _rname in enumerate(regs.names):
        m = (rid == i)
        if set(m.dims) == set(spatial) and list(m.dims) != spatial:
            m = m.transpose(*spatial)
        w = xr.where(m, 1.0, np.nan)
        out_list.append((da * w).mean(spatial, skipna=True))

    out = xr.concat(out_list, dim="region").assign_coords(region=regs.names)
    return out

def contains_xy_mask(geom, xs, ys):
    try:
        import shapely
        cx = getattr(shapely, "contains_xy", None)
        if callable(cx):
            return np.asarray(cx(geom, xs, ys), dtype=bool)
    except Exception:
        pass

    try:
        from shapely.vectorized import contains as vcontains
        return np.asarray(vcontains(geom, xs, ys), dtype=bool)
    except Exception:
        pass

    import shapely as _sh
    from shapely.prepared import prep
    pg = prep(geom)
    return np.array([pg.contains(_sh.Point(x, y)) for x, y in zip(xs, ys)], dtype=bool)

def fix_duplicate_dims_da(da: xr.DataArray) -> xr.DataArray:
    dims = list(da.dims)
    if len(dims) == len(set(dims)):
        return da
    seen, new_dims = {}, []
    for d in dims:
        seen[d] = seen.get(d, 0) + 1
        new_dims.append(d if seen[d] == 1 else f"{d}{seen[d]-1}")
    da.dims = tuple(new_dims)
    return da

def fix_duplicate_dims(ds: xr.Dataset) -> xr.Dataset:
    for name, var in ds.variables.items():
        dims = list(getattr(var, "dims", ()))
        if len(dims) == len(set(dims)):
            continue
        seen, new_dims = {}, []
        for d in dims:
            seen[d] = seen.get(d, 0) + 1
            new_dims.append(d if seen[d] == 1 else f"{d}{seen[d]-1}")
        ds[name].dims = tuple(new_dims)
    return ds

def align_mask_to_da(mask_da: xr.DataArray, da: xr.DataArray) -> xr.DataArray:
    spatial = [d for d in da.dims if d != "time"]
    if len(spatial) != 2:
        raise RuntimeError(f"Expected 2 spatial dims, got {spatial}")

    mask_da = mask_da.squeeze(drop=True)
    mask_da = fix_duplicate_dims_da(mask_da)
    if mask_da.ndim != 2:
        raise RuntimeError(f"Mask must be 2D; got dims={mask_da.dims}, shape={mask_da.shape}")

    mdims = list(mask_da.dims)

    rename_map = {}
    used_sd = set()
    for md in mdims:
        if md in spatial:
            rename_map[md] = md
            used_sd.add(md)

    remaining_md = [md for md in mdims if md not in rename_map]
    remaining_sd = [sd for sd in spatial if sd not in used_sd]

    for md in remaining_md:
        matched = None
        for sd in remaining_sd:
            if mask_da.sizes[md] == da.sizes[sd]:
                matched = sd
                break
        if matched is None:
            raise RuntimeError(
                f"Could not map mask dim '{md}' (size={mask_da.sizes[md]}) "
                f"to any of {remaining_sd} with sizes {[da.sizes[s] for s in remaining_sd]}"
            )
        rename_map[md] = matched
        remaining_sd.remove(matched)

    m = mask_da.rename(rename_map)

    if set(m.dims) == set(spatial) and list(m.dims) != spatial:
        m = m.transpose(*spatial)

    if (m.sizes.get(spatial[0], -1) != da.sizes[spatial[0]]) or (m.sizes.get(spatial[1], -1) != da.sizes[spatial[1]]):
        raise RuntimeError(
            f"Mask shape {(m.sizes.get(spatial[0]), m.sizes.get(spatial[1]))} "
            f"does not match data grid {(da.sizes[spatial[0]], da.sizes[spatial[1]])}"
        )
    return m

def safe_open_zarr(p: Path):
    try:
        return xr.open_zarr(p, consolidated=True)
    except Exception:
        return xr.open_zarr(p, consolidated=False)

def pick_var(ds, candidates):
    for v in candidates:
        if v in ds.data_vars:
            return v
    cl = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cl:
            return v
    return None

def daily_temp_df_from_series(dataset_name, region_name, time_index, tmean=None, tmin=None, tmax=None):
    df = pd.DataFrame({"time": pd.to_datetime(time_index)})
    if tmean is None:
        df["tmean_c"] = np.nan
    else:
        df["tmean_c"] = np.asarray(tmean, dtype=float)

    if tmin is None:
        df["tmin_c"] = np.nan
    else:
        df["tmin_c"] = np.asarray(tmin, dtype=float)

    if tmax is None:
        df["tmax_c"] = np.nan
    else:
        df["tmax_c"] = np.asarray(tmax, dtype=float)

    df["dataset"] = dataset_name
    df["region"] = region_name
    return df[["dataset", "region", "time", "tmean_c", "tmin_c", "tmax_c"]]

def combine_daily_frames(frames):
    frames = [f for f in frames if f is not None and not f.empty]
    if not frames:
        return pd.DataFrame(columns=["dataset", "region", "time", "tmean_c", "tmin_c", "tmax_c"])
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"])
    out = out.sort_values(["dataset", "region", "time"]).drop_duplicates(
        subset=["dataset", "region", "time"], keep="first"
    ).reset_index(drop=True)
    return out

def native_region_df_to_daily(native_df, dataset_name):
    """
    native_df columns: region, time, temp_c
    Returns daily dataframe with tmean/tmin/tmax
    """
    frames = []
    if native_df.empty:
        return pd.DataFrame(columns=["dataset", "region", "time", "tmean_c", "tmin_c", "tmax_c"])

    for region in sorted(native_df["region"].unique()):
        sub = (
            native_df[native_df["region"] == region][["time", "temp_c"]]
            .dropna()
            .sort_values("time")
            .drop_duplicates(subset=["time"], keep="first")
            .set_index("time")["temp_c"]
        )
        if sub.empty:
            continue

        daily = sub.resample("1D").agg(["mean", "min", "max"])
        daily = daily.reset_index()

        frames.append(
            daily_temp_df_from_series(
                dataset_name=dataset_name,
                region_name=region,
                time_index=daily["time"],
                tmean=daily["mean"],
                tmin=daily["min"],
                tmax=daily["max"],
            )
        )

    return combine_daily_frames(frames)

regions_gdf = load_regions_gdf()


HOT_DAY_THRESH_C = 30.0   # 86 F
FREEZE_THRESH_C  = 0.0    # 32 F

def annual_mean_metric(series_c: pd.Series, min_days=MIN_DAYS_PER_WY):
    series_c = series_c.dropna().sort_index()
    if series_c.empty:
        return pd.Series(dtype=float)

    groups = water_year_index(series_c.index)
    counts = series_c.groupby(groups).count()
    metric = series_c.groupby(groups).mean()
    metric = metric[counts >= min_days]
    return filter_year_range(metric)

def annual_count_metric(series_c: pd.Series, threshold_c: float, op: str, min_days=MIN_DAYS_PER_WY):
    series_c = series_c.dropna().sort_index()
    if series_c.empty:
        return pd.Series(dtype=float)

    groups = water_year_index(series_c.index)
    counts = series_c.groupby(groups).count()

    if op == "gt":
        metric = (series_c > threshold_c).groupby(groups).sum().astype(float)
    elif op == "lt":
        metric = (series_c < threshold_c).groupby(groups).sum().astype(float)
    else:
        raise ValueError(op)

    metric = metric[counts >= min_days]
    return filter_year_range(metric)

def summer_mean_max_metric(series_c: pd.Series, min_days=MIN_SUMMER_DAYS):
    """
    JJA of year Y belongs to WY Y too, so same WY index is fine.
    """
    series_c = series_c.dropna().sort_index()
    if series_c.empty:
        return pd.Series(dtype=float)

    summer = series_c[series_c.index.month.isin([6, 7, 8])]
    if summer.empty:
        return pd.Series(dtype=float)

    groups = water_year_index(summer.index)
    counts = summer.groupby(groups).count()
    metric = summer.groupby(groups).mean()
    metric = metric[counts >= min_days]
    return filter_year_range(metric)

def summarize_region_daily_df(dataset_name: str, region_name: str, region_df: pd.DataFrame):
    region_df = region_df.sort_values("time").drop_duplicates(subset=["time"], keep="first").copy()
    if region_df.empty:
        return None, None

    region_df["time"] = pd.to_datetime(region_df["time"])

    tmean_c = region_df.set_index("time")["tmean_c"].dropna().sort_index()
    tmin_c  = region_df.set_index("time")["tmin_c"].dropna().sort_index()
    tmax_c  = region_df.set_index("time")["tmax_c"].dropna().sort_index()

    # fallback for annual average only when tmean is missing but tmin+tmax exist
    if tmean_c.empty and (not tmin_c.empty) and (not tmax_c.empty):
        idx = tmin_c.index.intersection(tmax_c.index)
        if len(idx) > 0:
            tmean_c = ((tmin_c.loc[idx] + tmax_c.loc[idx]) / 2.0).sort_index()

    annual_avg_c = annual_mean_metric(tmean_c)
    annual_avg_daily_min_c = annual_mean_metric(tmin_c)
    annual_avg_daily_max_c = annual_mean_metric(tmax_c)
    summer_avg_max_c = summer_mean_max_metric(tmax_c)
    annual_days_max_above_30c = annual_count_metric(tmax_c, HOT_DAY_THRESH_C, "gt")
    annual_days_min_below_0c  = annual_count_metric(tmin_c, FREEZE_THRESH_C, "lt")

    union_years = sorted(
        set(annual_avg_c.index) |
        set(annual_avg_daily_min_c.index) |
        set(annual_avg_daily_max_c.index) |
        set(summer_avg_max_c.index) |
        set(annual_days_max_above_30c.index) |
        set(annual_days_min_below_0c.index)
    )

    annual_rows = []
    for y in union_years:
        annual_rows.append({
            "dataset": dataset_name,
            "region": region_name,
            "year": int(y),
            "annual_avg_c": float(annual_avg_c.get(y, np.nan)),
            "annual_avg_daily_min_c": float(annual_avg_daily_min_c.get(y, np.nan)),
            "annual_avg_daily_max_c": float(annual_avg_daily_max_c.get(y, np.nan)),
            "summer_avg_max_c": float(summer_avg_max_c.get(y, np.nan)),
            "days_max_above_30c": float(annual_days_max_above_30c.get(y, np.nan)),
            "days_min_below_0c": float(annual_days_min_below_0c.get(y, np.nan)),
        })

    start_candidates = []
    end_candidates = []
    for s in [tmean_c, tmin_c, tmax_c]:
        if not s.empty:
            start_candidates.append(s.index.min())
            end_candidates.append(s.index.max())

    summary_row = {
        "dataset": dataset_name,
        "region": region_name,
        "record_start": min(start_candidates) if start_candidates else pd.NaT,
        "record_end": max(end_candidates) if end_candidates else pd.NaT,
        "n_years_annual_avg": int(len(annual_avg_c)),
        "annual_average_c": float(annual_avg_c.mean()) if not annual_avg_c.empty else np.nan,
        "annual_average_daily_min_c": float(annual_avg_daily_min_c.mean()) if not annual_avg_daily_min_c.empty else np.nan,
        "annual_average_daily_max_c": float(annual_avg_daily_max_c.mean()) if not annual_avg_daily_max_c.empty else np.nan,
        "average_summer_jja_max_c": float(summer_avg_max_c.mean()) if not summer_avg_max_c.empty else np.nan,
        "annual_average_days_max_above_30c": float(annual_days_max_above_30c.mean()) if not annual_days_max_above_30c.empty else np.nan,
        "annual_average_days_min_below_0c": float(annual_days_min_below_0c.mean()) if not annual_days_min_below_0c.empty else np.nan,
    }

    return summary_row, pd.DataFrame(annual_rows)

def summarize_dataset_daily_df(dataset_name: str, daily_df: pd.DataFrame):
    summary_rows = []
    annual_parts = []

    if daily_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    for region in sorted(daily_df["region"].unique()):
        sub = daily_df[daily_df["region"] == region].copy()
        row, ann = summarize_region_daily_df(dataset_name, region, sub)
        if row is not None:
            summary_rows.append(row)
        if ann is not None and not ann.empty:
            annual_parts.append(ann)

    summary_df = pd.DataFrame(summary_rows)
    annual_df = pd.concat(annual_parts, ignore_index=True) if annual_parts else pd.DataFrame()
    return summary_df, annual_df


def year_from_path(p):
    m = re.search(r"_(\d{4})\.nc$", str(p))
    return int(m.group(1)) if m else None

def good_file(p, var):
    try:
        ds = xr.open_dataset(p)
        ok = (var in ds.data_vars) and (ds.sizes.get("time", 0) > 0)
        ds.close()
        return ok
    except Exception:
        return False

def open_daymet_year(path_nc: str, var: str) -> xr.Dataset:
    ds = xr.open_dataset(path_nc, engine="netcdf4")
    ds = _ensure_time_name(ds)
    if var not in ds:
        ds.close()
        raise KeyError(f"{path_nc} missing var={var}")
    if "x" in ds.coords:
        ds = ds.sortby("x")
    if "y" in ds.coords:
        ds = ds.sortby("y")
    if "time" in ds.coords:
        tt = pd.to_datetime(ds["time"].values)
        _, idx = np.unique(tt, return_index=True)
        ds = ds.isel(time=np.sort(idx))
    return ds[[var]]

def load_daymet_daily(regions_gdf):
    dataset_name = "Daymet v4 (local tmean/tmin/tmax)"
    print(f"[{dataset_name}] loading...")

    tmin_files = sorted(glob.glob(DAYMET_TMIN_GLOB))
    tmax_files = sorted(glob.glob(DAYMET_TMAX_GLOB))

    tmin_good = {year_from_path(f): f for f in tmin_files if year_from_path(f) is not None and good_file(f, "tmin")}
    tmax_good = {year_from_path(f): f for f in tmax_files if year_from_path(f) is not None and good_file(f, "tmax")}

    years = sorted([y for y in range(YEAR_MIN - 1, YEAR_MAX + 1) if y in tmin_good and y in tmax_good])
    if not years:
        print(f"[{dataset_name}] no overlapping usable years")
        return pd.DataFrame()

    dsmin_list, dsmax_list = [], []
    frames = []
    try:
        for y in years:
            dsmin_list.append(open_daymet_year(tmin_good[y], "tmin"))
            dsmax_list.append(open_daymet_year(tmax_good[y], "tmax"))

        ds_min = xr.concat(dsmin_list, dim="time", join="outer", coords="minimal", compat="override").sortby("time")
        ds_max = xr.concat(dsmax_list, dim="time", join="outer", coords="minimal", compat="override").sortby("time")

        tmin = ensure_time_sorted_unique(ds_min["tmin"])
        tmax = ensure_time_sorted_unique(ds_max["tmax"])
        tmin2, tmax2 = xr.align(tmin, tmax, join="inner")

        tmin2 = to_celsius(tmin2).sel(time=slice(GLOBAL_START, GLOBAL_END))
        tmax2 = to_celsius(tmax2).sel(time=slice(GLOBAL_START, GLOBAL_END))
        tmean2 = ((tmin2 + tmax2) / 2.0)

        rm_tmin = region_mean(tmin2, grid_ds=ds_min, regions_gdf=regions_gdf).load()
        rm_tmax = region_mean(tmax2, grid_ds=ds_max, regions_gdf=regions_gdf).load()
        rm_tmean = region_mean(tmean2, grid_ds=ds_min, regions_gdf=regions_gdf).load()

        for region in REGION_NAMES:
            s_mean = rm_tmean.sel(region=region).to_series()
            s_min  = rm_tmin.sel(region=region).to_series()
            s_max  = rm_tmax.sel(region=region).to_series()

            df_r = daily_temp_df_from_series(
                dataset_name=dataset_name,
                region_name=region,
                time_index=s_mean.index,
                tmean=s_mean.values,
                tmin=s_min.reindex(s_mean.index).values,
                tmax=s_max.reindex(s_mean.index).values,
            )

            df_r = df_r[~df_r["time"].dt.year.isin([])].copy()

            wy = water_year_index(df_r["time"])
            df_r = df_r[~pd.Index(wy).isin(DAYMET_SKIP_WY)].copy()
            frames.append(df_r)

        return combine_daily_frames(frames)

    finally:
        for d in dsmin_list:
            try:
                d.close()
            except Exception:
                pass
        for d in dsmax_list:
            try:
                d.close()
            except Exception:
                pass

def _detect_prism_temp_var(ds):
    candidates = ["tmean", "tavg", "tmean_c", "tmean_degC", "Tmean", "temp", "tas"]
    for v in candidates:
        if v in ds.data_vars:
            return v
    for v in ds.data_vars:
        vl = v.lower()
        if "tmean" in vl or "tavg" in vl or ("temp" in vl and "mean" in vl):
            return v
    return None

def _parse_prism_zarr_year_range(p: Path):
    m = re.match(r"^(\d{4})-\d{2}-\d{2}_(\d{4})-\d{2}-\d{2}_daily_4km_PRISM_data\.zarr$", p.name)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

def _prepare_prism_da(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        tdim = next((d for d in da.dims if "time" in d.lower() or "day" in d.lower() or "date" in d.lower()), None)
        if tdim:
            da = da.rename({tdim: "time"})
    da = ensure_time_sorted_unique(da)
    return da

def _month_keys_from_time(time_vals) -> set[str]:
    tt = pd.to_datetime(time_vals)
    return set(tt.to_period("M").astype(str).tolist())

def _extract_prism_nc_from_zip(zpath: Path, cache_dir: Path) -> Path | None:
    cache_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        members = zf.namelist()
        nc_members = [m for m in members if m.lower().endswith(".nc")]
        if not nc_members:
            return None

        inner = nc_members[0]
        out = cache_dir / f"{zpath.stem}.nc"
        if out.exists():
            return out

        zf.extract(inner, path=cache_dir)
        extracted = cache_dir / inner
        out.parent.mkdir(parents=True, exist_ok=True)
        if extracted.resolve() != out.resolve():
            if out.exists():
                out.unlink()
            extracted.rename(out)
        return out

def load_prism_daily(regions_gdf):
    dataset_name = "PRISM (tmean 4km)"
    print(f"[{dataset_name}] loading...")

    prism_rm_parts = []
    loaded_months = set()
    target_months = set(pd.date_range(GLOBAL_START, GLOBAL_END, freq="MS").strftime("%Y-%m").tolist())

    prism_zarrs_all = sorted([p for p in PRISM_TEMP_DIR.glob("*_daily_4km_PRISM_data.zarr") if p.is_dir()])
    zarr_info = []
    for p in prism_zarrs_all:
        yr = _parse_prism_zarr_year_range(p)
        if yr is not None:
            zarr_info.append((p, yr[0], yr[1]))

    zarr_info = sorted(zarr_info, key=lambda x: (x[1], -x[2]))
    selected_zarrs = []
    for p, ys, ye in zarr_info:
        contained = any((ys >= s and ye <= e) for _, s, e in selected_zarrs)
        if not contained:
            selected_zarrs.append((p, ys, ye))

    for p, ys, ye in selected_zarrs:
        try:
            ds = safe_open_zarr(p)
            v = _detect_prism_temp_var(ds)
            if v is None:
                try:
                    ds.close()
                except Exception:
                    pass
                continue

            da = _prepare_prism_da(ds[v])
            if "time" not in da.dims:
                try:
                    ds.close()
                except Exception:
                    pass
                continue

            da = to_celsius(da).sel(time=slice(GLOBAL_START, GLOBAL_END))
            if da.sizes.get("time", 0) == 0:
                try:
                    ds.close()
                except Exception:
                    pass
                continue

            rm = region_mean(da, grid_ds=ds, regions_gdf=regions_gdf)
            rm = ensure_time_sorted_unique(rm).load()
            prism_rm_parts.append(rm)
            loaded_months |= _month_keys_from_time(rm["time"].values)

            try:
                ds.close()
            except Exception:
                pass

        except Exception as e:
            print(f"[{dataset_name}] failed zarr {p.name}: {repr(e)}")

    prism_zips = sorted(PRISM_TEMP_DIR.glob(PRISM_ZIP_GLOB))
    cache_dir = PRISM_TEMP_DIR / "_prism_nc_cache"
    ym_re = re.compile(r"_(\d{6})$")

    for zp in prism_zips:
        m = ym_re.search(zp.stem)
        if not m:
            continue

        ym = m.group(1)
        month_key = f"{ym[:4]}-{ym[4:]}"
        if month_key not in target_months or month_key in loaded_months:
            continue

        nc_path = _extract_prism_nc_from_zip(zp, cache_dir)
        if nc_path is None:
            continue

        try:
            ds = xr.open_dataset(nc_path, engine="netcdf4")
            ds = _ensure_time_name(ds)

            v = _detect_prism_temp_var(ds)
            if v is None:
                ds.close()
                continue

            da = _prepare_prism_da(ds[v])
            if "time" not in da.dims:
                ds.close()
                continue

            da = to_celsius(da).sel(time=slice(GLOBAL_START, GLOBAL_END))
            if da.sizes.get("time", 0) == 0:
                ds.close()
                continue

            rm = region_mean(da, grid_ds=ds, regions_gdf=regions_gdf)
            rm = ensure_time_sorted_unique(rm).load()
            prism_rm_parts.append(rm)
            loaded_months |= _month_keys_from_time(rm["time"].values)
            ds.close()

        except Exception as e:
            print(f"[{dataset_name}] failed zip {zp.name}: {repr(e)}")

    if not prism_rm_parts:
        print(f"[{dataset_name}] no usable data")
        return pd.DataFrame()

    prism_rm = xr.concat(prism_rm_parts, dim="time", join="outer", coords="minimal", compat="override")
    prism_rm = ensure_time_sorted_unique(prism_rm).sel(time=slice(GLOBAL_START, GLOBAL_END))

    frames = []
    for region in REGION_NAMES:
        s_mean = prism_rm.sel(region=region).to_series()
        frames.append(
            daily_temp_df_from_series(
                dataset_name=dataset_name,
                region_name=region,
                time_index=s_mean.index,
                tmean=s_mean.values,
                tmin=None,
                tmax=None,
            )
        )

    return combine_daily_frames(frames)


def build_ucla_region_masks(coord_file: Path, regions_gdf):
    g = xr.open_dataset(coord_file, engine="netcdf4")
    lat2d = g["lat2d"].squeeze(drop=True).astype("float64")
    lon2d = g["lon2d"].squeeze(drop=True).astype("float64")
    g.close()

    regions_ll = regions_gdf.to_crs("EPSG:4326")
    lonv = lon2d.values.ravel()
    latv = lat2d.values.ravel()
    ny, nx = lon2d.shape

    out = {}
    for rname in REGION_NAMES:
        geom = regions_ll.loc[regions_ll["Name"] == rname, "geometry"].iloc[0]
        m_flat = contains_xy_mask(geom, lonv, latv)
        mask2d = m_flat.reshape(ny, nx).astype(bool)
        out[rname] = xr.DataArray(mask2d, dims=lon2d.dims, coords=lon2d.coords)
    return out

def load_ucla_daily(regions_gdf):
    dataset_name = "UCLA ERA5 d02 (daily t2)"
    print(f"[{dataset_name}] loading...")

    files = sorted(glob.glob(UCLA_T2_GLOB))
    if not files:
        print(f"[{dataset_name}] no files")
        return pd.DataFrame()
    if not UCLA_COORD_FILE.exists():
        print(f"[{dataset_name}] missing coord file")
        return pd.DataFrame()

    ucla_region_masks = build_ucla_region_masks(UCLA_COORD_FILE, regions_gdf)

    frames = []
    ds_u = None
    try:
        ds_u = xr.open_mfdataset(files, combine="by_coords", parallel=False, join="outer")
        ds_u = _ensure_time_name(ds_u)

        v = pick_var(ds_u, ["t2", "T2", "t2m", "tas", "temp"])
        if v is None:
            print(f"[{dataset_name}] temperature variable not found")
            return pd.DataFrame()

        da = to_celsius(ensure_time_sorted_unique(ds_u[v]).sel(time=slice(GLOBAL_START, GLOBAL_END)))
        spatial = [d for d in da.dims if d != "time"]
        if len(spatial) != 2:
            raise RuntimeError(f"Unexpected UCLA spatial dims: {spatial}")

        for region in REGION_NAMES:
            m = align_mask_to_da(ucla_region_masks[region], da)
            bm_daily = da.where(m).mean(dim=spatial, skipna=True).load()
            s_mean = bm_daily.to_series()

            frames.append(
                daily_temp_df_from_series(
                    dataset_name=dataset_name,
                    region_name=region,
                    time_index=s_mean.index,
                    tmean=s_mean.values,
                    tmin=None,
                    tmax=None,
                )
            )

        return combine_daily_frames(frames)

    finally:
        if ds_u is not None:
            try:
                ds_u.close()
            except Exception:
                pass


def load_hrrr06_daily(regions_gdf):
    dataset_name = "HRRR06 (t2m)"
    print(f"[{dataset_name}] loading...")

    monthly = sorted(P_HRRR_DIR.glob("*.zarr")) if P_HRRR_DIR.exists() else []
    monthly = [p for p in monthly if re.match(r"^\d{4}-\d{2}.*\.zarr$", p.name)]
    if not monthly:
        print(f"[{dataset_name}] no monthly zarrs")
        return pd.DataFrame()

    need_prefixes = set(pd.date_range(GLOBAL_START, GLOBAL_END, freq="MS").strftime("%Y-%m").tolist())
    parts = sorted([p for p in monthly if p.name[:7] in need_prefixes], key=lambda p: p.name[:7])
    if not parts:
        print(f"[{dataset_name}] no overlapping months")
        return pd.DataFrame()

    native_frames = []
    for p in parts:
        ds = None
        try:
            ds = safe_open_zarr(p)
            if "t2m" not in ds:
                try:
                    ds.close()
                except Exception:
                    pass
                continue

            da = to_celsius(ensure_time_sorted_unique(ds["t2m"])).sel(time=slice(GLOBAL_START, GLOBAL_END))
            if da.sizes.get("time", 0) == 0:
                try:
                    ds.close()
                except Exception:
                    pass
                continue

            rm_native = region_mean(da, grid_ds=ds, regions_gdf=regions_gdf).load()

            for region in REGION_NAMES:
                s = rm_native.sel(region=region).to_series().dropna().sort_index()
                if s.empty:
                    continue
                df_r = pd.DataFrame({
                    "region": region,
                    "time": pd.to_datetime(s.index),
                    "temp_c": s.values,
                })
                native_frames.append(df_r)

            try:
                ds.close()
            except Exception:
                pass

        except Exception as e:
            print(f"[{dataset_name}] failed {p.name}: {repr(e)}")
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if not native_frames:
        return pd.DataFrame()

    native_df = pd.concat(native_frames, ignore_index=True)
    daily_df = native_region_df_to_daily(native_df, dataset_name)

    wy = water_year_index(daily_df["time"])
    daily_df = daily_df[~pd.Index(wy).isin(HRRR_SKIP_WY)].copy()
    return combine_daily_frames([daily_df])


def load_pnnl_daily(regions_gdf):
    dataset_name = "PNNL (historical T2)"
    print(f"[{dataset_name}] loading...")

    if not PNNL_HIST.exists():
        print(f"[{dataset_name}] missing PNNL_HIST")
        return pd.DataFrame()
    if not PNNL_GEO.exists():
        print(f"[{dataset_name}] missing PNNL_GEO")
        return pd.DataFrame()

    try:
        import regionmask
    except Exception as e:
        print(f"[{dataset_name}] regionmask missing: {e}")
        return pd.DataFrame()

    geo = xr.open_dataset(PNNL_GEO)
    lat_name = next((c for c in ["XLAT_M", "XLAT", "lat", "LAT"] if c in geo), None)
    lon_name = next((c for c in ["XLONG_M", "XLONG", "lon", "LON"] if c in geo), None)
    if lat_name is None or lon_name is None:
        geo.close()
        print(f"[{dataset_name}] could not find geo lat/lon")
        return pd.DataFrame()

    lat2 = _to_2d(geo[lat_name]).astype("float64")
    lon2 = _to_2d(geo[lon_name]).astype("float64")

    regs = regionmask.Regions(
        outlines=list(regions_gdf.geometry.values),
        names=list(regions_gdf["Name"].astype(str).values),
        numbers=list(range(len(regions_gdf))),
        name="Skagit_3HUC8",
    )
    rid = regs.mask(lon2, lat2)

    pnnl_masks_np = {r: np.asarray((rid == i).data, dtype=bool) for i, r in enumerate(regs.names)}
    rid_dims = rid.dims
    rid_coords = {d: rid[d] for d in rid.dims if d in rid.coords}
    geo.close()

    native_frames = []

    for wy_i in range(YEAR_MIN, YEAR_MAX + 1):
        files = []
        files += sorted(glob.glob(str(PNNL_HIST / f"{wy_i-1}" / "*T2*.nc")))
        files += sorted(glob.glob(str(PNNL_HIST / f"{wy_i}"   / "*T2*.nc")))
        if not files:
            continue

        def _preprocess(ds):
            return ds[["T2"]] if "T2" in ds.data_vars else ds

        dsp = None
        try:
            dsp = xr.open_mfdataset(
                files,
                combine="by_coords",
                engine="netcdf4",
                parallel=False,
                preprocess=_preprocess,
                chunks={"time": 24 * 7},
                join="outer",
            )
            dsp = _ensure_time_name(dsp)
            dsp = fix_duplicate_dims(dsp)

            if "T2" not in dsp:
                dsp.close()
                continue

            start = f"{wy_i-1}-10-01"
            end = f"{wy_i}-09-30"

            t2 = ensure_time_sorted_unique(dsp["T2"]).sel(time=slice(start, end))
            if t2.sizes.get("time", 0) == 0:
                dsp.close()
                continue

            t2 = to_celsius(t2)
            spatial = [d for d in t2.dims if d != "time"]
            if len(spatial) != 2:
                dsp.close()
                raise RuntimeError(f"[{dataset_name}] unexpected spatial dims: {spatial}")

            for region in REGION_NAMES:
                m_geo = xr.DataArray(pnnl_masks_np[region], dims=rid_dims, coords=rid_coords)
                m = align_mask_to_da(m_geo, t2)
                basin_series = t2.where(m).mean(dim=spatial, skipna=True).load()
                s = basin_series.to_series().dropna().sort_index()
                if s.empty:
                    continue

                native_frames.append(pd.DataFrame({
                    "region": region,
                    "time": pd.to_datetime(s.index),
                    "temp_c": s.values,
                }))

            dsp.close()

        except Exception as e:
            print(f"[{dataset_name}] WY{wy_i} failed: {repr(e)}")
            if dsp is not None:
                try:
                    dsp.close()
                except Exception:
                    pass

    if not native_frames:
        return pd.DataFrame()

    native_df = pd.concat(native_frames, ignore_index=True)
    daily_df = native_region_df_to_daily(native_df, dataset_name)
    return combine_daily_frames([daily_df])


def load_conus404_daily(regions_gdf):
    dataset_name = "CONUS404"
    print(f"[{dataset_name}] loading...")

    try:
        import intake
    except Exception as e:
        print(f"[{dataset_name}] intake missing: {e}")
        return pd.DataFrame()

    try:
        ds_cat = intake.open_catalog(
            "https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml"
        )
        conus_url = ds_cat["conus404-catalog"].path
        conus_cat = intake.open_catalog(conus_url)

        # Prefer daily OSN entries
        candidate_keys = [k for k in list(conus_cat) if ("daily" in k.lower() and "osn" in k.lower())]
        if not candidate_keys:
            candidate_keys = [k for k in list(conus_cat) if "osn" in k.lower()]

        if not candidate_keys:
            print(f"[{dataset_name}] no usable OSN entries found")
            return pd.DataFrame()

        last_err = None

        for used_key in candidate_keys:
            ds_c = None
            try:
                print(f"[{dataset_name}] trying catalog entry: {used_key}")
                ds_c = conus_cat[used_key].to_dask()
                ds_c = fix_duplicate_dims(ds_c)

                if "lon" in ds_c:
                    ds_c["lon"] = fix_duplicate_dims_da(ds_c["lon"])
                if "lat" in ds_c:
                    ds_c["lat"] = fix_duplicate_dims_da(ds_c["lat"])

                v = pick_var(ds_c, ["t2m", "tas", "T2", "temp", "temperature"])
                if v is None:
                    print(f"[{dataset_name}] {used_key}: no temp variable found")
                    if ds_c is not None:
                        try:
                            ds_c.close()
                        except Exception:
                            pass
                    continue

                # subset first, then compute
                da = ensure_time_sorted_unique(ds_c[v]).sel(time=slice(GLOBAL_START, GLOBAL_END))
                da = to_celsius(da)
                da = da.chunk({"time": 365})

                rm = region_mean(da, grid_ds=ds_c, regions_gdf=regions_gdf)
                rm = ensure_time_sorted_unique(rm).load()

                frames = []
                for region in REGION_NAMES:
                    s_mean = rm.sel(region=region).to_series()
                    frames.append(
                        daily_temp_df_from_series(
                            dataset_name=f"CONUS404 ({used_key})",
                            region_name=region,
                            time_index=s_mean.index,
                            tmean=s_mean.values,
                            tmin=None,
                            tmax=None,
                        )
                    )

                if ds_c is not None:
                    try:
                        ds_c.close()
                    except Exception:
                        pass

                return combine_daily_frames(frames)

            except Exception as e:
                last_err = e
                print(f"[{dataset_name}] {used_key} failed: {repr(e)}")
                if ds_c is not None:
                    try:
                        ds_c.close()
                    except Exception:
                        pass

        print(f"[{dataset_name}] all candidate entries failed")
        if last_err is not None:
            print(f"[{dataset_name}] final error: {repr(last_err)}")
        return pd.DataFrame()

    except Exception as e:
        print(f"[{dataset_name}] failed before read: {repr(e)}")
        return pd.DataFrame()

all_daily = []
all_summary = []
all_annual = []

def run_one(dataset_name, loader_func):
    print(f"\n{'='*72}\nRunning {dataset_name}\n{'='*72}")
    t0 = time.time()
    daily_df = loader_func(regions_gdf)
    if daily_df is None or daily_df.empty:
        print(f"[{dataset_name}] no data returned")
        return

    summary_df, annual_df = summarize_dataset_daily_df(dataset_name, daily_df)
    if not summary_df.empty:
        all_summary.append(summary_df)
    if not annual_df.empty:
        all_annual.append(annual_df)
    all_daily.append(daily_df)

    print(f"[{dataset_name}] done in {(time.time() - t0)/60.0:.2f} min")

if RUN_DATASETS["DAYMET"]:
    run_one("Daymet v4 (local tmean/tmin/tmax)", load_daymet_daily)

if RUN_DATASETS["PRISM"]:
    run_one("PRISM (tmean 4km)", load_prism_daily)

if RUN_DATASETS["UCLA"]:
    run_one("UCLA ERA5 d02 (daily t2)", load_ucla_daily)

if RUN_DATASETS["HRRR06"]:
    run_one("HRRR06 (t2m)", load_hrrr06_daily)

if RUN_DATASETS["PNNL"]:
    run_one("PNNL (historical T2)", load_pnnl_daily)

if RUN_DATASETS["CONUS404"]:
    run_one("CONUS404", load_conus404_daily)

if not all_summary:
    raise RuntimeError("No dataset produced output.")

daily_all = combine_daily_frames(all_daily)
summary_df = pd.concat(all_summary, ignore_index=True)
annual_df = pd.concat(all_annual, ignore_index=True)


annual_df = annual_df.sort_values(["region", "dataset", "year"]).reset_index(drop=True)
summary_df = summary_df.sort_values(["region", "dataset"]).reset_index(drop=True)

annual_df.to_csv(ANNUAL_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)

print("Saved annual metrics ->", ANNUAL_CSV)
print("Saved summary metrics ->", SUMMARY_CSV)


metric_name_map = {
    "annual_average_c": "Annual Average (°C)",
    "annual_average_daily_min_c": "Annual Average Daily Min (°C)",
    "annual_average_daily_max_c": "Annual Average Daily Max (°C)",
    "average_summer_jja_max_c": "Average Summer (June - Aug) Max Temp (°C)",
    "annual_average_days_max_above_30c": "Annual Average Days with Max Temp Above 30°C",
    "annual_average_days_min_below_0c": "Annual Average Days with Min Temp Below 0°C",
}

metric_order = [
    "annual_average_c",
    "annual_average_daily_min_c",
    "annual_average_daily_max_c",
    "average_summer_jja_max_c",
    "annual_average_days_max_above_30c",
    "annual_average_days_min_below_0c",
]

for region in REGION_NAMES:
    sub = summary_df[summary_df["region"] == region].copy()
    if sub.empty:
        continue

    wide = sub.set_index("dataset")[metric_order].T
    wide.index = [metric_name_map[x] for x in wide.index]

    print("\n" + "=" * 100)
    print(f"REGION: {region}")
    print("=" * 100)
    display(wide.round(2))


display(
    annual_df.sort_values(["region", "dataset", "year"]).reset_index(drop=True).round(2)
)


for region in REGION_NAMES:
    sub = summary_df[summary_df["region"] == region].copy()
    if sub.empty:
        continue

    wide = sub.set_index("dataset")[metric_order].T
    wide.index = [metric_name_map[x] for x in wide.index]

    out_csv = OUT / f"summary_table_{region.replace(' ', '_')}_{YEAR_MIN}_{YEAR_MAX}.csv"
    wide.to_csv(out_csv)
    print("Saved region summary table ->", out_csv)


Running Daymet v4 (local tmean/tmin/tmax)
[Daymet v4 (local tmean/tmin/tmax)] loading...
[Daymet v4 (local tmean/tmin/tmax)] done in 1.30 min

Running PRISM (tmean 4km)
[PRISM (tmean 4km)] loading...
[PRISM (tmean 4km)] done in 0.05 min

Running UCLA ERA5 d02 (daily t2)
[UCLA ERA5 d02 (daily t2)] loading...
[UCLA ERA5 d02 (daily t2)] done in 0.42 min

Running HRRR06 (t2m)
[HRRR06 (t2m)] loading...
[HRRR06 (t2m)] done in 0.95 min

Running PNNL (historical T2)
[PNNL (historical T2)] loading...
[PNNL (historical T2)] done in 18.09 min

Running CONUS404
[CONUS404] loading...
[CONUS404] trying catalog entry: conus404-daily-diagnostic-osn
[CONUS404] conus404-daily-diagnostic-osn: no temp variable found
[CONUS404] trying catalog entry: conus404-daily-osn
[CONUS404] conus404-daily-osn failed: RuntimeError("Zstd decompression error: b'Data corruption detected'")
[CONUS404] trying catalog entry: conus404-daily-ba-osn
[CONUS404] conus404-daily-ba-osn: no temp variable found
[CONUS404] trying cat

dataset,CONUS404,Daymet v4 (local tmean/tmin/tmax),HRRR06 (t2m),PNNL (historical T2),PRISM (tmean 4km),UCLA ERA5 d02 (daily t2)
Annual Average (°C),5.55,4.65,5.50,3.08,5.13,3.72
Annual Average Daily Min (°C),NaN,0.04,2.72,0.10,NaN,NaN
Annual Average Daily Max (°C),NaN,9.27,8.37,7.27,NaN,NaN
Average Summer (June - Aug) Max Temp (°C),NaN,18.32,17.18,18.47,NaN,NaN
Annual Average Days with Max Temp Above 30°C,NaN,0.68,0.30,1.85,NaN,NaN
Annual Average Days with Min Temp Below 0°C,NaN,184.52,135.80,188.49,NaN,NaN



REGION: Sauk


dataset,CONUS404,Daymet v4 (local tmean/tmin/tmax),HRRR06 (t2m),PNNL (historical T2),PRISM (tmean 4km),UCLA ERA5 d02 (daily t2)
Annual Average (°C),6.44,5.65,6.16,4.30,5.73,4.68
Annual Average Daily Min (°C),NaN,1.07,3.45,1.39,NaN,NaN
Annual Average Daily Max (°C),NaN,10.23,8.96,8.45,NaN,NaN
Average Summer (June - Aug) Max Temp (°C),NaN,18.87,17.25,19.21,NaN,NaN
Annual Average Days with Max Temp Above 30°C,NaN,1.12,0.40,3.23,NaN,NaN
Annual Average Days with Min Temp Below 0°C,NaN,161.48,118.90,161.26,NaN,NaN



REGION: Lower Skagit


dataset,CONUS404,Daymet v4 (local tmean/tmin/tmax),HRRR06 (t2m),PNNL (historical T2),PRISM (tmean 4km),UCLA ERA5 d02 (daily t2)
Annual Average (°C),10.3,8.74,9.64,8.87,8.87,8.39
Annual Average Daily Min (°C),NaN,4.12,6.79,5.87,NaN,NaN
Annual Average Daily Max (°C),NaN,13.37,12.70,13.48,NaN,NaN
Average Summer (June - Aug) Max Temp (°C),NaN,21.21,20.71,23.27,NaN,NaN
Annual Average Days with Max Temp Above 30°C,NaN,2.50,2.10,11.36,NaN,NaN
Annual Average Days with Min Temp Below 0°C,NaN,76.03,37.30,57.36,NaN,NaN


,dataset,region,year,annual_avg_c,annual_avg_daily_min_c,annual_avg_daily_max_c,summer_avg_max_c,days_max_above_30c,days_min_below_0c
0,CONUS404,Lower Skagit,1980,9.40,NaN,NaN,NaN,NaN,NaN
1,CONUS404,Lower Skagit,1981,10.43,NaN,NaN,NaN,NaN,NaN
2,CONUS404,Lower Skagit,1982,9.27,NaN,NaN,NaN,NaN,NaN
3,CONUS404,Lower Skagit,1983,10.03,NaN,NaN,NaN,NaN,NaN
4,CONUS404,Lower Skagit,1984,9.12,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
658,UCLA ERA5 d02 (daily t2),Upper Skagit,2020,3.61,NaN,NaN,NaN,NaN,NaN
659,UCLA ERA5 d02 (daily t2),Upper Skagit,2021,4.21,NaN,NaN,NaN,NaN,NaN
660,UCLA ERA5 d02 (daily t2),Upper Skagit,2022,4.32,NaN,NaN,NaN,NaN,NaN
661,UCLA ERA5 d02 (daily t2),Upper Skagit,2023,4.79,NaN,NaN,NaN,NaN,NaN


Saved region summary table -> /data0/balaji24/data/derived/temperature_summary_metrics/summary_table_Upper_Skagit_1980_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/temperature_summary_metrics/summary_table_Sauk_1980_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/temperature_summary_metrics/summary_table_Lower_Skagit_1980_2024.csv
